In [14]:
import os
import time
import json
import subprocess
import pyautogui
import socket
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: mps


### Loading the pretrained model

In [15]:
# Load the trained model
model_path = 'models/pokemon_model_lstm.pth'
model = torch.load(model_path)
model.load_state_dict()
model.eval()

# Load in annotation model
annotation_model_pth = "runs/detect/finalRun/weights/best.pt"
annotation_modelmodel = YOLO(annotation_model_pth)

# Data paths
states_dir = ""
actions_dir = ""
annotations_dir = ""

FileNotFoundError: [Errno 2] No such file or directory: 'models/pokemon_model_lstm.pth'

In [ ]:
# Setting model hyperparameters
gamma = 0.99            # Discount factor for future rewards
epsilon = 1.0           # Initial exploration rate
epsilon_decay = 0.995   # Decay rate for the exploration probability per episode
min_epsilon = 0.01      # Minimum exploration rate
learning_rate = 0.001   # Rate at which model changes parameters
num_episodes = 1000     # Number of episodes to engage in

# Action mapping for emulator
ACTION_MAP = {
    'A': 'x',
    'B': 'z',
    'X': 's',
    'Y': 'a',
    'Up': 'up',
    'Down': 'down',
    'Left': 'left',
    'Right': 'right'
}

#### Examining how epsilon (exploration rate) changes through episodes (training)

In [ ]:
# List to store epsilon values
epsilon_values = []

# Simulate epsilon decay over episodes
for episode in range(num_episodes):
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    epsilon_values.append(epsilon)

# Plotting the epsilon decay
plt.figure(figsize=(10, 6))
plt.plot(epsilon_values, label='Epsilon')
plt.xlabel('Episode')
plt.ylabel('Epsilon Value')
plt.title('Epsilon Decay Over Episodes')
plt.legend()
plt.grid(True)
plt.show()


### Connecting script to emulator (DeSmuME) via .lua script

In [ ]:
# Path to the DeSmuME executable, the ROM file, and desired state
# Won't be starting from scratch to avoid uneccessary cutscenes
desmume_executable = '/Applications/DeSmuME.app/Contents/MacOS/DeSmuME'
pokemon_rom = 'Platinum_Randomized.nds'
state_file = '/Users/scottpitcher/Library/Application Support/DeSmuME/0.9.13/States/Platinum_Randomized.ds4'
lua_script = 'env.lua'


# Start DeSmuME emulator
def open_emulator():
    subprocess.Popen([desmume_executable, pokemon_rom, '--load-lua-script', lua_script])
    time.sleep(5)

    pyautogui.hotkey('fn', 'F') # Full screen

    # Wait for the emulator to start (this is the amount of time until actions can be sent to emulator after loading screen)
    time.sleep(15) 
    
    # Load in the state
    key = f'4'
    time.sleep(0.2)
    pyautogui.press('x')
    time.sleep(0.2)
    pyautogui.press(key)
    print(f"Loaded state {key}")

open_emulator()

2024-06-25 13:51:14.330 DeSmuME[5409:7242163] WARNING: Secure coding is not enabled for restorable state! Enable secure coding by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState: and returning YES.


Loaded state 4


Microphone successfully inited.
DeSmuME 0.9.13 ARM64 NEON-A64

ROM game code: CPUE
ROM crc: A95C7305
ROM serial: NTR-CPUE-USA
ROM chipID: 00007FC2
ROM internal name: POKEMON PL
ROM developer: Nintendo

Load cheats: /Users/scottpitcher/Library/Application Support/DeSmuME/0.9.13/Cheats/Platinum_Randomized.dct
Added 2 cheat codes
Slot1 auto-selected device type: Retail MC+ROM
Slot2 auto-selected device type: None (0xFF)
BackupDevice: size = 4 Mbit
CPU mode: Interpreter
Already decrypted.
WIFI: MAC Address = 00:09:BF:9B:27:8F
WIFI: Emulation level is OFF.
Load cheats: /Users/scottpitcher/Library/Application Support/DeSmuME/0.9.13/Cheats/Platinum_Randomized.dct
Added 2 cheat codes
SoftRasterizer: Running using 8 additional threads. (Multithreading enabled.)
Slot1 auto-selected device type: Retail MC+ROM
Slot2 auto-selected device type: None (0xFF)
BackupDevice: size = 4 Mbit
CPU mode: Interpreter
Already decrypted.
WIFI: MAC Address = 00:09:BF:9B:27:8F
WIFI: Emulation level is OFF.
Slot 2: 

In [ ]:
# Functions to interact with the emulator

def send_command_to_lua(command):
    client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client.connect(("localhost", 12345))
    client.sendall(command.encode())
    response = client.recv(1024).decode()
    client.close()
    return response

def perform_action(action):
    if action in ACTION_MAP:
        response = send_command_to_lua(f"PERFORM_ACTION {ACTION_MAP[action]}")
        print(response)

def get_game_state(output_dir='game_state_screenshots', frame_num=0):
    response = send_command_to_lua("GET_STATE")
    print(response)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    screenshot = pyautogui.screenshot()
    frame_path = os.path.join(output_dir, f'frame_{frame_num}.png')
    screenshot.save(frame_path)
    print(f"Saved game state to {frame_path}")

### Defining Rewards

In [ ]:
def calculate_reward(state, action, next_state):
    # Define rewards based on game events (example logic)
    if 'battle' in state and 'battle' not in next_state:
        return 10  # Reward for winning a battle
    elif 'city' in next_state:
        return 5  # Reward for entering a city
    else:
        return -1  # Small penalty for other actions

# Example reward function with annotations (optional)
def calculate_reward_with_annotations(state, action, next_state, annotations):
    reward = calculate_reward(state, action, next_state)
    if 'desired_object' in annotations:
        reward += 2  # Additional reward for detecting desired object
    return reward

### Collecting Human Feedback

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_rlhf_model(epochs, model, criterion, optimizer, annotation_model):
    for epoch in range(epochs):
        state = get_game_state()
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        
        # Annotate the state using the annotation model
        annotations = annotation_model.predict(state_tensor)
        
        # Decide on action
        action = model(state_tensor)
        action_idx = action.argmax().item()
        action_key = list(ACTION_MAP.keys())[action_idx]
        
        # Execute action
        perform_action(action_key)
        
        # Get next state
        next_state = get_game_state()
        next_state_tensor = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0)
        
        # Calculate reward
        reward = calculate_reward(state, action, next_state)
        
        # Update model
        target = reward + gamma * model(next_state_tensor).max().item()
        loss = criterion(action, torch.tensor([target]))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item()}")

# Example usage
annotation_model = load_annotation_model()
train_rlhf_model(epochs=100, model=model, criterion=criterion, optimizer=optimizer, annotation_model=annotation_model)